# KNeighborsClassifier

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score)
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold, cross_validate

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [22]:
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
valid_df = pd.read_csv(os.path.join(data_dir, 'valid_data.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]
X_test  = test_df.drop(columns=[target_col])
y_test  = test_df[target_col]

###  Danh sách lưu kết quả để xuất CSV

In [23]:
results_list = []
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_kfold(model, name, X_data, y_data, cv_split):
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'auc': 'roc_auc'
    }
    cv_results = cross_validate(model, X_data, y_data, cv=cv_split, scoring=scoring, n_jobs=-1)
    res = {
        'Algorithm': name,
        'Split': 'KFold',
        'Accuracy':  round(cv_results['test_accuracy'].mean(), 2),
        'Precision': round(cv_results['test_precision'].mean(), 2),
        'Recall':    round(cv_results['test_recall'].mean(), 2),
        'F1-Score':  round(cv_results['test_f1'].mean(), 2),
        'AUC':       round(cv_results['test_auc'].mean(), 2)
    }
    results_list.append(res)
    print(f"[{name} | KFold] Acc: {res['Accuracy']} | F1: {res['F1-Score']} | AUC: {res['AUC']}")
    return res

def evaluate_model(model, name, X_split, y_split, split_name):
    y_pred = model.predict(X_split)
    y_prob = model.predict_proba(X_split)[:, 1]
    res = {
        'Algorithm': name,
        'Split': split_name,
        'Accuracy':  round(accuracy_score(y_split, y_pred), 2),
        'Precision': round(precision_score(y_split, y_pred), 2),
        'Recall':    round(recall_score(y_split, y_pred), 2),
        'F1-Score':  round(f1_score(y_split, y_pred), 2),
        'AUC':       round(roc_auc_score(y_split, y_prob), 2)
    }
    results_list.append(res)
    print(f"[{name} | {split_name}] Acc: {res['Accuracy']} | F1: {res['F1-Score']} | AUC: {res['AUC']}")
    return res

### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [24]:
print("--- V1: Baseline kNN ---")
v1_model = KNeighborsClassifier()  # mặc định k=5
evaluate_kfold(v1_model, "V1: kNN Baseline", X_train, y_train, kfold)
v1_model.fit(X_train, y_train)
evaluate_model(v1_model, "V1: kNN Baseline", X_valid, y_valid, "Validation")
evaluate_model(v1_model, "V1: kNN Baseline", X_test,  y_test,  "Test")

--- V1: Baseline kNN ---
[V1: kNN Baseline | KFold] Acc: 0.99 | F1: 0.99 | AUC: 1.0
[V1: kNN Baseline | Validation] Acc: 1.0 | F1: 0.99 | AUC: 1.0
[V1: kNN Baseline | Test] Acc: 1.0 | F1: 0.99 | AUC: 1.0


{'Algorithm': 'V1: kNN Baseline',
 'Split': 'Test',
 'Accuracy': 1.0,
 'Precision': 0.99,
 'Recall': 1.0,
 'F1-Score': 0.99,
 'AUC': 1.0}

### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [25]:
print("\n--- V2: GridSearchCV Tuning ---")
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
v2_model = grid_search.best_estimator_

print(f"Best Params: {grid_search.best_params_}")
evaluate_kfold(v2_model, "V2: kNN Tuned (GridSearch)", X_train, y_train, kfold)
evaluate_model(v2_model, "V2: kNN Tuned (GridSearch)", X_valid, y_valid, "Validation")
evaluate_model(v2_model, "V2: kNN Tuned (GridSearch)", X_test,  y_test,  "Test")


--- V2: GridSearchCV Tuning ---
Best Params: {'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'distance'}
[V2: kNN Tuned (GridSearch) | KFold] Acc: 1.0 | F1: 1.0 | AUC: 1.0
[V2: kNN Tuned (GridSearch) | Validation] Acc: 1.0 | F1: 1.0 | AUC: 1.0
[V2: kNN Tuned (GridSearch) | Test] Acc: 1.0 | F1: 1.0 | AUC: 1.0


{'Algorithm': 'V2: kNN Tuned (GridSearch)',
 'Split': 'Test',
 'Accuracy': 1.0,
 'Precision': 0.99,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

In [26]:
print("\n--- V3: Ensemble Learning (Stacking) ---")
base_estimators = [
    ('knn', v2_model),
    ('rf',  RandomForestClassifier(n_estimators=50, random_state=42))
]

v3_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=kfold
)
evaluate_kfold(v3_model, "V3: kNN Stacking Ensemble", X_train, y_train, kfold)
v3_model.fit(X_train, y_train)
evaluate_model(v3_model, "V3: kNN Stacking Ensemble", X_valid, y_valid, "Validation")
evaluate_model(v3_model, "V3: kNN Stacking Ensemble", X_test,  y_test,  "Test")


--- V3: Ensemble Learning (Stacking) ---


[V3: kNN Stacking Ensemble | KFold] Acc: 1.0 | F1: 1.0 | AUC: 1.0
[V3: kNN Stacking Ensemble | Validation] Acc: 1.0 | F1: 1.0 | AUC: 1.0
[V3: kNN Stacking Ensemble | Test] Acc: 1.0 | F1: 1.0 | AUC: 1.0


{'Algorithm': 'V3: kNN Stacking Ensemble',
 'Split': 'Test',
 'Accuracy': 1.0,
 'Precision': 0.99,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

### (5) Lưu kết quả

In [27]:
save_path = '../experiment/KNN/'
os.makedirs(save_path, exist_ok=True)

# Lưu CSV
df_results = pd.DataFrame(results_list)
csv_output = os.path.join(save_path, 'knn_evaluation_results.csv')
df_results.to_csv(csv_output, index=False)

print("\n" + "-" * 40)
print(f"Đã lưu CSV tại: {csv_output}")
print(f"Đã lưu models tại: {save_path}")
print("-" * 40)
display(df_results)


----------------------------------------
Đã lưu CSV tại: ../experiment/KNN/knn_evaluation_results.csv
Đã lưu models tại: ../experiment/KNN/
----------------------------------------


,Algorithm,Split,Accuracy,Precision,Recall,F1-Score,AUC
0,V1: kNN Baseline,KFold,0.99,0.99,1.0,0.99,1.0
1,V1: kNN Baseline,Validation,1.00,0.99,1.0,0.99,1.0
2,V1: kNN Baseline,Test,1.00,0.99,1.0,0.99,1.0
3,V2: kNN Tuned (GridSearch),KFold,1.00,0.99,1.0,1.00,1.0
4,V2: kNN Tuned (GridSearch),Validation,1.00,1.00,1.0,1.00,1.0
5,V2: kNN Tuned (GridSearch),Test,1.00,0.99,1.0,1.00,1.0
6,V3: kNN Stacking Ensemble,KFold,1.00,0.99,1.0,1.00,1.0
7,V3: kNN Stacking Ensemble,Validation,1.00,1.00,1.0,1.00,1.0
8,V3: kNN Stacking Ensemble,Test,1.00,0.99,1.0,1.00,1.0


In [28]:
!jupyter nbconvert --to html KNeighborsClassifier.ipynb

[NbConvertApp] Converting notebook KNeighborsClassifier.ipynb to html
[NbConvertApp] Writing 308246 bytes to KNeighborsClassifier.html
